# 확률분포 시 생성기 LoRA 파인튜닝 Colab 노트북

- base model: `Qwen/Qwen2.5-0.5B-Instruct`
- Colab T4 기준 QLoRA
- 입력: 사용자 경험 프롬프트
- 출력: 정확히 3줄 한국어 시
- 데이터셋: `messages` 형식 `train.jsonl`
- adapter와 merged model을 모두 Hugging Face Hub에 업로드


## 1. 라이브러리 설치


In [17]:
!pip -q install -U "transformers>=4.47.0" "accelerate>=1.2.0" "peft>=0.14.0" "datasets>=3.0.0" "trl>=0.12.0" "torchao>=0.16.0" bitsandbytes huggingface_hub sentencepiece


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 39.6 MB/s eta 0:00:00


## 2. Hugging Face 로그인


In [18]:
from huggingface_hub import login
login()


## 3. 기본 설정


In [19]:
import torch

BASE_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
ADAPTER_REPO_ID = "z-unghyun/poem-generator-lora-adapter"
MERGED_REPO_ID = "z-unghyun/poem-generator-merged"
OUTPUT_DIR = "./outputs/poem-generator-lora"
MERGED_OUTPUT_DIR = "./outputs/poem-generator-merged"

DATA_MODE = "local_upload"  # "local_upload" or "hub_dataset"
LOCAL_TRAIN_PATH = "train.jsonl"
HF_DATASET_ID = "z-unghyun/poem-generator-dataset"
HF_DATA_FILE = "train.jsonl"

LORA_TARGET_MODULES = ["q_proj", "v_proj"]
# LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]
# LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

print("cuda_available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("GPU가 잡히지 않았습니다. 런타임 유형을 T4 GPU로 바꿔주세요.")


cuda_available: True
gpu: Tesla T4


## 4. 데이터셋 불러오기


In [20]:
from datasets import load_dataset
from google.colab import files
from pathlib import Path

if DATA_MODE == "local_upload":
    if not Path(LOCAL_TRAIN_PATH).exists():
        uploaded = files.upload()
        if LOCAL_TRAIN_PATH not in uploaded:
            first_name = next(iter(uploaded.keys()))
            Path(LOCAL_TRAIN_PATH).write_bytes(uploaded[first_name])
            print(f"saved uploaded file as {LOCAL_TRAIN_PATH}")
    dataset = load_dataset("json", data_files=LOCAL_TRAIN_PATH, split="train")
else:
    dataset = load_dataset(HF_DATASET_ID, data_files=HF_DATA_FILE, split="train")

print(dataset)
print(dataset[0])


Dataset({
    features: ['messages'],
    num_rows: 60
})
{'messages': [{'role': 'system', 'content': '너는 경험을 3줄 한국어 시로 바꾸는 시 생성 모델이다.'}, {'role': 'user', 'content': '경험: 강가에서 위험한 선택을 하려는 사람을 붙잡지 못하고 바라봄'}, {'role': 'assistant', 'content': '강은 말보다 먼저 깊어지고\n붙잡던 소리는 물가에 남는다\n젖은 밤이 사람 하나를 접어 간다'}]}


## 5. 데이터셋 검수


In [21]:
def count_nonempty_lines(text: str) -> int:
    return len([line for line in text.splitlines() if line.strip()])

def validate_messages(example):
    messages = example.get("messages")
    if not isinstance(messages, list) or len(messages) != 3:
        return {"is_valid": False, "validation_reason": "messages_not_3_turns"}
    roles = [m.get("role") for m in messages]
    if roles != ["system", "user", "assistant"]:
        return {"is_valid": False, "validation_reason": f"invalid_roles:{roles}"}
    poem = messages[2].get("content", "")
    if count_nonempty_lines(poem) != 3:
        return {"is_valid": False, "validation_reason": f"assistant_not_three_lines:{count_nonempty_lines(poem)}"}
    return {"is_valid": True, "validation_reason": "ok"}

checked = dataset.map(validate_messages)
bad = checked.filter(lambda x: not x["is_valid"])
print("total:", len(checked))
print("bad:", len(bad))
if len(bad) > 0:
    print(bad[:5])
    raise ValueError("Invalid training rows found. Fix train.jsonl before training.")
dataset = checked.remove_columns(["is_valid", "validation_reason"])


total: 60
bad: 0


## 6. Chat template 적용


In [22]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def apply_chat_template(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = dataset.map(apply_chat_template, remove_columns=dataset.column_names)
print(dataset[0]["text"][:1000])


<|im_start|>system
너는 경험을 3줄 한국어 시로 바꾸는 시 생성 모델이다.<|im_end|>
<|im_start|>user
경험: 강가에서 위험한 선택을 하려는 사람을 붙잡지 못하고 바라봄<|im_end|>
<|im_start|>assistant
강은 말보다 먼저 깊어지고
붙잡던 소리는 물가에 남는다
젖은 밤이 사람 하나를 접어 간다<|im_end|>



## 7. 모델 로드: QLoRA 4bit


In [23]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


## 8. LoRA 설정과 SFTTrainer 생성

소량 데이터에서 실제 update 수가 부족하지 않도록 gradient accumulation을 낮추고 epoch를 늘린다.


In [24]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=LORA_TARGET_MODULES,
)

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_length=512,
    num_train_epochs=10,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    logging_steps=5,
    save_strategy="epoch",
    fp16=False,
    bf16=False,
    report_to="none",
    packing=False,
)

try:
    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=dataset,
        peft_config=lora_config,
        processing_class=tokenizer,
    )
except TypeError:
    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=dataset,
        peft_config=lora_config,
        tokenizer=tokenizer,
    )

trainer


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


## 9. 학습 실행 및 adapter 저장


In [25]:
trainer.train()
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("saved adapter:", OUTPUT_DIR)


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
5,4.470675
10,4.075909
15,3.551722
20,3.194694
25,2.887110
30,2.586777
35,2.393494
40,2.218691
45,2.172121
50,2.137485


saved adapter: ./outputs/poem-generator-lora


## 10. 학습된 adapter로 3줄 시 생성 테스트


In [26]:
def build_prompt(experience: str):
    messages = [
        {"role": "system", "content": "너는 경험을 3줄 한국어 시로 바꾸는 시 생성 모델이다. 설명 없이 시만 정확히 3줄로 쓴다."},
        {"role": "user", "content": f"경험: {experience}"},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def postprocess_poem(text: str):
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return "\n".join(lines[:3])

def validate_poem_text(text: str):
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if len(lines) == 3:
        return "valid", "ok"
    return "invalid_shown", f"not_three_lines:{len(lines)}"

test_inputs = [
    "야자 끝나고 비 오는 정류장에서 버스를 기다림",
    "새벽에 과제를 하다가 모니터 빛에 눈이 아픔",
    "카페에서 창업 아이디어를 정리하다가 갑자기 막힘",
    "혼자 여행지 술집에서 감자튀김과 맥주를 먹음",
    "AI 프로젝트를 하다가 언어가 망가지는 걸 봄",
]

model.eval()
for experience in test_inputs:
    prompt = build_prompt(experience)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=70,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
    raw_poem = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    poem = postprocess_poem(raw_poem)
    status, reason = validate_poem_text(poem)
    print("=" * 80)
    print("경험:", experience)
    print("원문 출력:")
    print(raw_poem)
    print("후처리 3줄 후보:")
    print(poem)
    print("lines:", len([line for line in poem.splitlines() if line.strip()]))
    print("validation:", status, reason)


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


경험: 야자 끝나고 비 오는 정류장에서 버스를 기다림
원문 출력:
버스가 잠재력으로 작은 도형을 만든다
정류장은 토름처럼 점점 밀어진다
우리 가족은 버스를 잠시 멈출 수 없다
후처리 3줄 후보:
버스가 잠재력으로 작은 도형을 만든다
정류장은 토름처럼 점점 밀어진다
우리 가족은 버스를 잠시 멈출 수 없다
lines: 3
validation: valid ok
경험: 새벽에 과제를 하다가 모니터 빛에 눈이 아픔
원문 출력:
모니터의 빛은 나의 신발에 부딪히고,
새벽은 내 눈이 흔들림으로 간다.
과학은 저의 몸을 편하게 만든다.
후처리 3줄 후보:
모니터의 빛은 나의 신발에 부딪히고,
새벽은 내 눈이 흔들림으로 간다.
과학은 저의 몸을 편하게 만든다.
lines: 3
validation: valid ok
경험: 카페에서 창업 아이디어를 정리하다가 갑자기 막힘
원문 출력:
카페의 냄새가 가파르게 펼쳐지고
창업 아이디어는 단숨에 끝나고
내 마음은 막 어려운 문을 놓치고
후처리 3줄 후보:
카페의 냄새가 가파르게 펼쳐지고
창업 아이디어는 단숨에 끝나고
내 마음은 막 어려운 문을 놓치고
lines: 3
validation: valid ok
경험: 혼자 여행지 술집에서 감자튀김과 맥주를 먹음
원문 출력:
수영장의 물은 감자 튀기고
맥주는 흔들림으로 뒤로 가는다
여기서 먹는 사람은 아무도 안한다
후처리 3줄 후보:
수영장의 물은 감자 튀기고
맥주는 흔들림으로 뒤로 가는다
여기서 먹는 사람은 아무도 안한다
lines: 3
validation: valid ok
경험: AI 프로젝트를 하다가 언어가 망가지는 걸 봄
원문 출력:
언어는 한 복잡한 문장에서 멀어진다
프로젝트는 시간이 지나고 기다림에 불가능하다
인공지능은 이제 새로운 푸른 날씨다
후처리 3줄 후보:
언어는 한 복잡한 문장에서 멀어진다
프로젝트는 시간이 지나고 기다림에 불가능하다
인공지능은 이제 새로운 푸른 날씨다
lines: 3
validation: valid ok


## 11. Adapter를 Hugging Face Hub에 업로드


In [27]:
trainer.model.push_to_hub(ADAPTER_REPO_ID)
tokenizer.push_to_hub(ADAPTER_REPO_ID)
print("adapter pushed:", ADAPTER_REPO_ID)


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  28%|##8       |  619kB / 2.18MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpoe19c5qv/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

No files have been modified since last commit. Skipping to prevent empty commit.


adapter pushed: z-unghyun/poem-generator-lora-adapter


## 12. Merged model 생성 및 업로드


In [28]:
import gc
from peft import PeftModel

try:
    del trainer
    del model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

base_model_for_merge = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
merged_model = PeftModel.from_pretrained(base_model_for_merge, OUTPUT_DIR)
merged_model = merged_model.merge_and_unload()
merged_model.save_pretrained(MERGED_OUTPUT_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_OUTPUT_DIR)
merged_model.push_to_hub(MERGED_REPO_ID)
tokenizer.push_to_hub(MERGED_REPO_ID)
print("merged model pushed:", MERGED_REPO_ID)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpb5ph_xq1/tokenizer.json:  70%|######9   | 7.99MB / 11.4MB            

merged model pushed: z-unghyun/poem-generator-merged


## 13. Merged model 로드 테스트


In [29]:
test_model = AutoModelForCausalLM.from_pretrained(
    MERGED_REPO_ID,
    dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
test_tokenizer = AutoTokenizer.from_pretrained(MERGED_REPO_ID, trust_remote_code=True)
if test_tokenizer.pad_token is None:
    test_tokenizer.pad_token = test_tokenizer.eos_token
experience = "밤새 만든 앱 화면을 아침에 다시 보니 전부 고치고 싶어짐"
messages = [
    {"role": "system", "content": "너는 경험을 3줄 한국어 시로 바꾸는 시 생성 모델이다."},
    {"role": "user", "content": f"경험: {experience}"},
]
prompt = test_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = test_tokenizer(prompt, return_tensors="pt").to(test_model.device)
with torch.no_grad():
    output_ids = test_model.generate(
        **inputs,
        max_new_tokens=70,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=test_tokenizer.eos_token_id,
    )
poem = test_tokenizer.decode(output_ids[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
print(poem)
print("lines:", len([line for line in poem.splitlines() if line.strip()]))


config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/2.51k [00:00<?, ?B/s]

화면은 밤마다 터무서운 속이어서
나는 새로운 프로그램을 시작하게 되어
전체를 뒤로 떠올린다.
lines: 3
